In [40]:
import sys
sys.path.append('..')
import torch
import soundfile as sf
import json
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
import numpy as np
import argparse
import random

from encoder import FMEncoder, compute_spectrogram_cqt
from fm_ddsp import fm_renderer, make_mod_matrix
from nnAudio.features import CQT2010v2
from generate_dataset import generate_dataset
from train import train_stage1

%load_ext autoreload
%autoreload 2

# Stage 1 — Single modulator, single carrier
    # Fixed f0=440, one modulator → one carrier, vary ratio of modulator and level. Encoder learns the relationship between ratio and sideband position.
# Stage 2 — Two operators, fixed algorithm
    # Still fixed f0, introduce a second modulator. Encoder learns to handle multiple simultaneous modulators.
# Stage 3 — Variable f0, fixed algorithm
    # Now vary f0 across MIDI range. Encoder learns pitch invariance — same timbre at different pitches should give same parameters.
# Stage 4 — Multiple algorithms
    # Introduce algorithm variety. Encoder learns to distinguish different routing topologies.
# Stage 5 — Full dataset
    # Everything varying simultaneously.

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [41]:
def stage1_params():
    ratio_choices = [0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 5.0, 6.0]
    # only use operator 2
    ratio_op2 = random.choice(ratio_choices)
    mod_level_op2 = torch.rand(1)
    return {
        'f0': 440.0,
        'algorithm': 'algo_2',
        'mod_values': torch.tensor([0,0,0,0,0,0,1], dtype=torch.float32),
        'ratios': torch.tensor([1,1,ratio_op2,1], dtype = torch.float32),
        'levels': torch.tensor([0,0,mod_level_op2, 1], dtype = torch.float32),
        'carrier_weights': torch.tensor([0.0, 0.0, 0.0, 1.0], dtype = torch.float32)
    }

In [42]:
save_dir_pc = "B:\\TrainingData\\stage1"
#save_dir = '../data/stage1',
args = argparse.Namespace(
    n_examples = 40000,
    save_dir = save_dir_pc,
    Fs = 16000,
    duration = 1.0,
    seed = 127,
    overwrite=True
)
generate_dataset(args, param_fn = stage1_params)

Low pass filter created, time used = 0.0010 seconds
num_octave =  7
No early downsampling is required, downsample_factor =  1
Early downsampling filter created,                         time used = 0.0000 seconds
CQT kernels created, time used = 0.0010 seconds
Generating example 00000/40000
Generating example 04000/40000
Generating example 08000/40000
Generating example 12000/40000
Generating example 16000/40000
Generating example 20000/40000
Generating example 24000/40000
Generating example 28000/40000
Generating example 32000/40000
Generating example 36000/40000
Generating example 39999/40000
Finished Generating Examples


In [43]:
# train stage 1
args = argparse.Namespace(
    data_dir = save_dir_pc,
    Fs = 16000,
    duration = 1.0,
    lr = 0.0001,
    n_epochs = 30,
    resume = None,
    start_epoch = 0
)
train_stage1(args)

Using Cuda? True
Low pass filter created, time used = 0.0010 seconds
num_octave =  7
No early downsampling is required, downsample_factor =  1
Early downsampling filter created,                         time used = 0.0000 seconds
CQT kernels created, time used = 0.0010 seconds
Epoch 0/30:
Average epoch loss: 0.38293602607250216
Average weight change: 0.00174674391746521
Epoch 1/30:
Average epoch loss: 0.3587324967384338
Average weight change: 0.0013456494780257344
Epoch 2/30:
Average epoch loss: 0.3384890845060349
Average weight change: 0.0009055024711415172
Epoch 3/30:
Average epoch loss: 0.32565870094299315
Average weight change: 0.0009615165763534606
Epoch 4/30:
Average epoch loss: 0.3163920472025871
Average weight change: 0.0016166669083759189
Epoch 5/30:
Average epoch loss: 0.30768758215904235
Average weight change: 0.0015993457054719329
Epoch 6/30:
Average epoch loss: 0.299592822098732
Average weight change: 0.001459755701944232
Epoch 7/30:
Average epoch loss: 0.29484190833568574
